On-Policy Distillation (OPD) is a post-training/distillation setup where the student model generates its own rollouts, and then a teacher model gives feedback on those student-generated states/tokens.

## 1. Standard Distillation / KD

In normal knowledge distillation:

```
prompt x
teacher generates or provides target distribution
student trains on teacher data
```

**Example:**

```
Prompt:
    "The capital of France is"

Teacher / dataset response:
    "Paris."

Student trains to imitate:
    "Paris."
```

This is often **off-policy** from the student's perspective because the training
states are produced by the teacher/dataset, not by the student's own sampled behavior.

The problem is:

```
At inference time:
    student sees its own prefixes

During training:
    student mostly sees teacher/dataset prefixes
```

This mismatch is called **exposure bias**.

---

## 2. On-Policy Distillation

In OPD, the **student generates first**.

```
Prompt:
    "The capital of France is"

Student rollout:
    "Pariss is a city..."

Teacher then evaluates the student rollout.
```

The teacher may provide:

```
token-level probabilities
corrections
step-level feedback
preference feedback
reward/verifier scores
```

Then the student updates on those states. So the training states are:

```
student-generated states
```

not just teacher-generated states. That is why it is called **on-policy**.

---

## 3. Why OPD is Useful

OPD helps because the student learns to **recover from its own mistakes**.

**Example:**

```
Student prefix:
    "The capital of France is Pariss"

Teacher can say:
    this is low probability / incorrect / should correct toward Paris
```

In standard KD, the student may never train on the prefix `"Pariss"` because
the teacher would not generate it.

So OPD is useful when:

```
student behavior differs from teacher behavior
student rollouts are long
errors compound over time
we want dense teacher feedback instead of sparse rewards
```

The verl OPD documentation contrasts OPD with RLVR by noting that OPD can provide
**dense continuous token-level supervision** rather than sparse outcome-level rewards.

## OPD Two modes:

## Decoupled mode :
verl's docs explicitly call this decoupled mode, with three policies $\pi_{\text{rollout}}, \pi_{\text{old}}, \pi_\theta$, and IS correction from rollout to old.

## In by pass mode :
π_old = π_rollout, πθ    = trainable policy

In [76]:
# ============================================================
# Teacher-forced verl-close PG-OPD with TIS
# Corrected: teacher_signal is anchored at π_old and frozen
# Aggregation: seq-mean-token-mean
# ============================================================

import copy
import numpy as np
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [77]:
# 0) Load student and teacher
# ------------------------------------------------------------

base_policy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Current trainable student policy: πθ
student_policy = copy.deepcopy(base_policy).to(device)

# Keep dropout off for demo clarity.
# Gradients still flow because requires_grad=True.
student_policy.eval()
for p in student_policy.parameters():
    p.requires_grad_(True)

# Frozen teacher policy: ν
# In real OPD, this is usually a stronger / larger model.
teacher_policy = copy.deepcopy(base_policy).to(device).eval()
for p in teacher_policy.parameters():
    p.requires_grad_(False)

print("Loaded:", model_name, "on", device)

Loaded: sshleifer/tiny-gpt2 on cpu


In [78]:
# ------------------------------------------------------------
# 1) Teacher-forced prompt + completion encoding
# ------------------------------------------------------------

def encode_prompt_response(tokenizer, prompt_text: str, response_text: str):
    """
    Teacher-forced prompt + response encoding.

    Returns:
        input_ids:    [1, seq_len]
        prompt_len:   number of prompt tokens
        response_len: number of response tokens
    """
    prompt_ids = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"]  # [1, prompt_len]

    full_ids = tokenizer(
        prompt_text + response_text,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"]  # [1, full_len]

    prompt_len = prompt_ids.shape[1]
    response_len = full_ids.shape[1] - prompt_len

    assert response_len > 0, "Response must add at least one token."

    return full_ids, prompt_len, response_len

In [79]:
# ------------------------------------------------------------
# 2) Selected-token log-probs for response tokens
# ------------------------------------------------------------

def token_logprobs_for_response(
    model,
    tokenizer,
    prompt: str,
    completion: str,
    require_grad: bool,
    temperature: float = 1.0,
):
    """
    Teacher-forced scoring of completion tokens.

    For completion token at absolute position pos,
    probability comes from logits at pos - 1.

    Returns:
        response_token_ids: [T]
        selected_logps:     [T]
    """
    input_ids, prompt_len, _ = encode_prompt_response(
        tokenizer,
        prompt,
        completion,
    )

    model_device = next(model.parameters()).device
    input_ids = input_ids.to(model_device)

    ctx = torch.enable_grad() if require_grad else torch.no_grad()

    with ctx:
        out = model(input_ids, use_cache=False)

        logits = out.logits / temperature
        logp_all = F.log_softmax(logits, dim=-1)

        token_ids = input_ids[0]  # [seq_len]
        seq_len = token_ids.shape[0]

        response_token_ids = token_ids[prompt_len:]  # [T]

        # response token at pos is predicted by logits at pos - 1
        prev_pos = torch.arange(
            prompt_len - 1,
            seq_len - 1,
            device=model_device,
        )  # [T]

        lp_rows = logp_all[0, prev_pos, :]  # [T, vocab]

        selected_logps = lp_rows.gather(
            dim=1,
            index=response_token_ids.unsqueeze(1),
        ).squeeze(1)  # [T]

    return response_token_ids, selected_logps

In [80]:
def rollout_is_weight_token_level(
    logp_old: torch.Tensor,
    logp_rollout: torch.Tensor,
    threshold: float = 2.0,
    batch_normalize: bool = False,
):
    """
    Strict truncated importance sampling:

        raw_ratio = exp(logp_old - logp_rollout)
        weight    = min(raw_ratio, threshold)

    Stable log-space implementation:

        log_weight = min(log_ratio, log(threshold))
        weight     = exp(log_weight)
    """
    log_ratio = logp_old - logp_rollout

    log_threshold = torch.log(
        torch.tensor(
            threshold,
            device=log_ratio.device,
            dtype=log_ratio.dtype,
        )
    )

    log_weight = torch.clamp(
        log_ratio,
        max=log_threshold,
    )

    weight = torch.exp(log_weight)

    if batch_normalize:
        weight = weight / (weight.mean() + 1e-8)

    return weight.detach()

In [81]:
# ------------------------------------------------------------
# 4) Collect one teacher-forced OPD item
# ------------------------------------------------------------

@torch.no_grad()
def collect_one_teacher_forced_opd_item(
    rollout_policy,
    old_policy,
    teacher_policy,
    tokenizer,
    prompt: str,
    completion: str,
    teacher_temperature: float,
    rollout_is_threshold: float,
    rollout_is_batch_normalize: bool,
):
    """
    Collect fixed rollout item using teacher forcing.

    We treat `completion` as already sampled from π_rollout.

    Roles:
        π_rollout:
            behavior policy that produced/scored rollout tokens

        π_old:
            proximal/training-old policy

        ν:
            frozen teacher policy

    Stored:
        logp_rollout
        logp_old
        logp_teacher
        teacher_signal = sg(logp_teacher - logp_old)
        rollout_is_weight = min(π_old / π_rollout, C)
    """
    ids_rollout, logp_rollout = token_logprobs_for_response(
        rollout_policy,
        tokenizer,
        prompt,
        completion,
        require_grad=False,
        temperature=1.0,
    )

    ids_old, logp_old = token_logprobs_for_response(
        old_policy,
        tokenizer,
        prompt,
        completion,
        require_grad=False,
        temperature=1.0,
    )

    ids_teacher, logp_teacher = token_logprobs_for_response(
        teacher_policy,
        tokenizer,
        prompt,
        completion,
        require_grad=False,
        temperature=teacher_temperature,
    )

    assert torch.equal(ids_rollout, ids_old), (
        "Token mismatch between rollout policy and old policy."
    )
    assert torch.equal(ids_rollout, ids_teacher), (
        "Token mismatch between rollout policy and teacher policy."
    )

    rollout_is_weight = rollout_is_weight_token_level(
        logp_old=logp_old,
        logp_rollout=logp_rollout,
        threshold=rollout_is_threshold,
        batch_normalize=rollout_is_batch_normalize,
    )

    # IMPORTANT FIX:
    # Teacher signal is anchored at π_old and frozen at rollout collection time.
    # It should NOT be recomputed against moving πθ during the update.
    teacher_signal = (logp_teacher - logp_old).detach()

    return {
        "prompt": prompt,
        "completion": completion,
        "response_token_ids": ids_rollout.detach().cpu(),
        "logp_rollout": logp_rollout.detach().cpu(),
        "logp_old": logp_old.detach().cpu(),
        "logp_teacher": logp_teacher.detach().cpu(),
        "teacher_signal": teacher_signal.detach().cpu(),
        "rollout_is_weight": rollout_is_weight.detach().cpu()
    }


In [82]:
# ------------------------------------------------------------
# 5) Collect teacher-forced OPD batch/list
# ------------------------------------------------------------

@torch.no_grad()
def collect_teacher_forced_opd_batch(
    rollout_policy,
    old_policy,
    teacher_policy,
    tokenizer,
    prompt: str,
    completions: list[str],
    teacher_temperature: float,
    rollout_is_threshold: float,
    rollout_is_batch_normalize: bool,
):
    """
    Collect multiple fixed completions for the same prompt.

    Aggregation later will be:
        mean over tokens within each completion,
        then mean over completions.
    """
    rollout_items = []

    for completion in completions:
        item = collect_one_teacher_forced_opd_item(
            rollout_policy=rollout_policy,
            old_policy=old_policy,
            teacher_policy=teacher_policy,
            tokenizer=tokenizer,
            prompt=prompt,
            completion=completion,
            teacher_temperature=teacher_temperature,
            rollout_is_threshold=rollout_is_threshold,
            rollout_is_batch_normalize=rollout_is_batch_normalize,
        )

        rollout_items.append(item)

    return rollout_items

In [83]:
# ------------------------------------------------------------
# 6) Per-sequence PG-OPD objective (policy gradient - On policy distillation)
# ------------------------------------------------------------

def pg_opd_objective_for_one_sequence(
    student_policy,
    tokenizer,
    rollout_item: dict,
    clip_ratio_low: float,
    clip_ratio_high: float,
    require_grad: bool,
):
    """
    Compute one sequence's PG-OPD objective.

    For one fixed completion:

        PPO update ratio:
            r_t(θ) = πθ(y_t | s_t) / π_old(y_t | s_t)

        Rollout IS correction:
            w_t = min(π_old(y_t | s_t) / π_rollout(y_t | s_t), C)

        Frozen PG-OPD teacher signal:
            A_t = sg(log ν(y_t | s_t) - log π_old(y_t | s_t))

        PPO-style clipped objective:
            min(r_t * A_t, clip(r_t) * A_t)

        Rollout-corrected objective:
            w_t * min(...)

        Sequence objective:
            mean over tokens in this completion
    """
    prompt = rollout_item["prompt"]
    completion = rollout_item["completion"]

    token_ids_old = rollout_item["response_token_ids"].to(device)
    logp_old = rollout_item["logp_old"].to(device)
    teacher_signal = rollout_item["teacher_signal"].to(device)
    rollout_is_weight = rollout_item["rollout_is_weight"].to(device)

    ids_current, logp_current = token_logprobs_for_response(
        student_policy,
        tokenizer,
        prompt,
        completion,
        require_grad=require_grad,
        temperature=1.0,
    )

    assert torch.equal(ids_current, token_ids_old), (
        "Current student token mismatch with rollout batch."
    )

    # PPO-style current/old ratio:
    ratio = torch.exp(logp_current - logp_old)

    ratio_clipped = torch.clamp(
        ratio,
        min=1.0 - clip_ratio_low,
        max=1.0 + clip_ratio_high,
    )

    surr1 = ratio * teacher_signal
    surr2 = ratio_clipped * teacher_signal

    surrogate = torch.minimum(surr1, surr2)

    # TIS / rollout correction:
    weighted_surrogate = rollout_is_weight * surrogate

    # Correct OPD aggregation for one sequence:
    # first average over tokens inside the sequence.
    sequence_objective = weighted_surrogate.mean()

    with torch.no_grad():
        clipfrac = (
            ((ratio > 1.0 + clip_ratio_high) | (ratio < 1.0 - clip_ratio_low))
            .float()
            .mean()
            .item()
        )

        seq_stats = {
            "sequence_objective": float(sequence_objective.detach().cpu()),
            "response_len": int(logp_current.numel()),
            "ratio_mean": float(ratio.detach().mean().cpu()),
            "ratio_min": float(ratio.detach().min().cpu()),
            "ratio_max": float(ratio.detach().max().cpu()),
            "clipfrac": clipfrac,
            "teacher_signal_mean": float(teacher_signal.mean().cpu()),
            "teacher_signal_min": float(teacher_signal.min().cpu()),
            "teacher_signal_max": float(teacher_signal.max().cpu()),
            "rollout_is_weight_mean": float(rollout_is_weight.mean().cpu()),
            "rollout_is_weight_min": float(rollout_is_weight.min().cpu()),
            "rollout_is_weight_max": float(rollout_is_weight.max().cpu()),
        }

    return sequence_objective, seq_stats

In [84]:
# ------------------------------------------------------------
# 7) Batch-level objective: seq-mean-token-mean
# ------------------------------------------------------------

def pg_opd_batch_objective_seq_mean_token_mean(
    student_policy,
    tokenizer,
    rollout_items: list[dict],
    clip_ratio_low: float,
    clip_ratio_high: float,
    require_grad: bool,
):
    """
    Correct OPD aggregation:

        objective =
            mean_i [
                mean_t objective_{i,t}
            ]

    This gives each completion equal weight,
    regardless of completion length.
    """
    sequence_objectives = []
    sequence_stats = []

    for item in rollout_items:
        seq_obj, seq_stat = pg_opd_objective_for_one_sequence(
            student_policy=student_policy,
            tokenizer=tokenizer,
            rollout_item=item,
            clip_ratio_low=clip_ratio_low,
            clip_ratio_high=clip_ratio_high,
            require_grad=require_grad,
        )

        sequence_objectives.append(seq_obj)
        sequence_stats.append(seq_stat)

    objective = torch.stack(sequence_objectives).mean()
    loss = -objective

    with torch.no_grad():
        response_lens = [s["response_len"] for s in sequence_stats]
        clipfracs = [s["clipfrac"] for s in sequence_stats]
        ratio_means = [s["ratio_mean"] for s in sequence_stats]
        is_means = [s["rollout_is_weight_mean"] for s in sequence_stats]
        teacher_signal_means = [s["teacher_signal_mean"] for s in sequence_stats]

        batch_stats = {
            "loss": float(loss.detach().cpu()),
            "objective": float(objective.detach().cpu()),
            "num_sequences": len(rollout_items),
            "response_lens": response_lens,
            "mean_response_len": float(np.mean(response_lens)),
            "mean_clipfrac": float(np.mean(clipfracs)),
            "mean_ratio": float(np.mean(ratio_means)),
            "mean_rollout_is_weight": float(np.mean(is_means)),
            "mean_teacher_signal": float(np.mean(teacher_signal_means)),
            "per_sequence_objective": [
                s["sequence_objective"] for s in sequence_stats
            ],
            "per_sequence_stats": sequence_stats,
        }

    return loss, batch_stats

In [85]:
# ------------------------------------------------------------
# 8) OPD update step
# ------------------------------------------------------------

def opd_pg_update_step_teacher_forced_verl_close(
    student_policy,
    optimizer,
    tokenizer,
    rollout_items: list[dict],
    clip_ratio_low: float,
    clip_ratio_high: float,
):
    """
    One PG-OPD update on fixed teacher-forced rollout items.

    Current student πθ recomputes log-probs with gradients.
    Rollout tensors are fixed/detached.
    """
    loss, stats = pg_opd_batch_objective_seq_mean_token_mean(
        student_policy=student_policy,
        tokenizer=tokenizer,
        rollout_items=rollout_items,
        clip_ratio_low=clip_ratio_low,
        clip_ratio_high=clip_ratio_high,
        require_grad=True,
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return stats

In [86]:
# ------------------------------------------------------------
# 9) Eval helper
# ------------------------------------------------------------

@torch.no_grad()
def opd_pg_eval_stats_teacher_forced_verl_close(
    student_policy,
    tokenizer,
    rollout_items: list[dict],
    clip_ratio_low: float,
    clip_ratio_high: float,
):
    """
    Compute the same stats without updating.
    """
    loss, stats = pg_opd_batch_objective_seq_mean_token_mean(
        student_policy=student_policy,
        tokenizer=tokenizer,
        rollout_items=rollout_items,
        clip_ratio_low=clip_ratio_low,
        clip_ratio_high=clip_ratio_high,
        require_grad=False,
    )

    return stats


In [87]:
# ------------------------------------------------------------
# 10) Settings
# ------------------------------------------------------------

# This is verl-style actor PPO epochs over one fixed sampled/rollout batch.
# This is NOT an OPD-specific "num_iterations".
# It comes from the actor PPO update machinery.
actor_ppo_epochs = 2

# Teacher temperature is only for this toy demo because teacher and student
# start from the same tiny checkpoint. In real OPD, teacher is usually different.
teacher_temperature = 0.7

# Rollout correction / TIS setting.
# This is C in min(π_old / π_rollout, C).
rollout_is_threshold = 2.0
rollout_is_batch_normalize = False

# PPO-style current/old clipping parameters.
clip_ratio_low = 0.2
clip_ratio_high = 0.28

lr = 1e-5
optimizer = torch.optim.Adam(student_policy.parameters(), lr=lr)


# ------------------------------------------------------------
# 11) Fixed teacher-forced examples
# ------------------------------------------------------------

examples = [
    {
        "prompt": "The capital of France is",
        "completions": [
            " Paris.",
            " London.",
        ],
    },
    {
        "prompt": "2 + 2 =",
        "completions": [
            " 4.",
            " 5.",
        ],
    },
]


# ------------------------------------------------------------
# 12) Main teacher-forced verl-close PG-OPD flow
# ------------------------------------------------------------

for example_idx, ex in enumerate(examples, start=1):

    print("\n" + "=" * 80)
    print(f"TEACHER-FORCED VERL-CLOSE PG-OPD EXAMPLE {example_idx}")
    print("=" * 80)

    prompt = ex["prompt"]
    completions = ex["completions"]

    print("Prompt:", repr(prompt))
    print("Completions:")
    for c in completions:
        print("   ", repr(c))

    # --------------------------------------------------------
    # A) Create policy roles
    # --------------------------------------------------------
    # π_rollout: behavior policy that generated/scored rollout tokens
    rollout_policy = copy.deepcopy(student_policy).to(device).eval()
    for p in rollout_policy.parameters():
        p.requires_grad_(False)

    # π_old: proximal/training-old anchor
    old_policy = copy.deepcopy(student_policy).to(device).eval()
    for p in old_policy.parameters():
        p.requires_grad_(False)

    print("\n[1] Policy roles")
    print("    π_rollout: frozen behavior policy for rollout tokens")
    print("    π_old:     frozen proximal/training-old policy")
    print("    πθ:        current trainable student_policy")
    print("    ν:         frozen teacher_policy")
    print("    In this HF demo, π_rollout and π_old are identical.")
    print("    In verl/vLLM-style systems, they may differ.")

    # --------------------------------------------------------
    # B) Collect teacher-forced rollout items ONCE
    # --------------------------------------------------------
    print("\n[2] Collect teacher-forced OPD rollout items once")

    rollout_items = collect_teacher_forced_opd_batch(
        rollout_policy=rollout_policy,
        old_policy=old_policy,
        teacher_policy=teacher_policy,
        tokenizer=tokenizer,
        prompt=prompt,
        completions=completions,
        teacher_temperature=teacher_temperature,
        rollout_is_threshold=rollout_is_threshold,
        rollout_is_batch_normalize=rollout_is_batch_normalize,
    )

    for i, item in enumerate(rollout_items):
        print(f"    item {i}:")
        print("        completion:", repr(item["completion"]))
        print("        response_len:", int(item["response_token_ids"].numel()))
        print("        teacher_signal mean/min/max:",
              float(item["teacher_signal"].mean()),
              float(item["teacher_signal"].min()),
              float(item["teacher_signal"].max()))
        print("        rollout_is_weight mean/min/max:",
              float(item["rollout_is_weight"].mean()),
              float(item["rollout_is_weight"].min()),
              float(item["rollout_is_weight"].max()))

    # --------------------------------------------------------
    # C) Before actor update
    # --------------------------------------------------------
    print("\n[3] Before actor PPO-style PG-OPD update")

    before = opd_pg_eval_stats_teacher_forced_verl_close(
        student_policy=student_policy,
        tokenizer=tokenizer,
        rollout_items=rollout_items,
        clip_ratio_low=clip_ratio_low,
        clip_ratio_high=clip_ratio_high,
    )

    print("    loss:", before["loss"])
    print("    objective:", before["objective"])
    print("    num_sequences:", before["num_sequences"])
    print("    response_lens:", before["response_lens"])
    print("    mean_response_len:", before["mean_response_len"])
    print("    mean_ratio:", before["mean_ratio"])
    print("    mean_rollout_is_weight:", before["mean_rollout_is_weight"])
    print("    mean_teacher_signal:", before["mean_teacher_signal"])
    print("    mean_clipfrac:", before["mean_clipfrac"])
    print("    per_sequence_objective:", before["per_sequence_objective"])

    # --------------------------------------------------------
    # D) Actor PPO epochs on this same fixed rollout batch
    # --------------------------------------------------------
    print("\n[4] Actor PPO-style PG-OPD update loop")
    print("    Teacher-forced scoring, no generate() call.")
    print("    Fixed rollout batch is reused across actor_ppo_epochs.")
    print("    actor_ppo_epochs:", actor_ppo_epochs)
    print("")
    print("    Fixed quantities for this rollout batch:")
    print("        logp_old from π_old")
    print("        teacher_signal = sg(log ν - log π_old)")
    print("        rollout_is_weight = min(π_old / π_rollout, C)")
    print("")
    print("    Recomputed each actor epoch:")
    print("        logp_current from πθ")
    print("        ratio = πθ / π_old")
    print("")
    print("    Uses seq-mean-token-mean aggregation:")
    print("        mean over tokens within each completion, then mean over completions.")

    for actor_epoch in range(1, actor_ppo_epochs + 1):

        stats = opd_pg_update_step_teacher_forced_verl_close(
            student_policy=student_policy,
            optimizer=optimizer,
            tokenizer=tokenizer,
            rollout_items=rollout_items,
            clip_ratio_low=clip_ratio_low,
            clip_ratio_high=clip_ratio_high,
        )

        print(f"\n    actor PPO epoch {actor_epoch}")
        print("        loss:", stats["loss"])
        print("        objective:", stats["objective"])
        print("        mean_ratio:", stats["mean_ratio"])
        print("        mean_rollout_is_weight:", stats["mean_rollout_is_weight"])
        print("        mean_teacher_signal:", stats["mean_teacher_signal"])
        print("        mean_clipfrac:", stats["mean_clipfrac"])
        print("        per_sequence_objective:", stats["per_sequence_objective"])

    # --------------------------------------------------------
    # E) After actor update
    # --------------------------------------------------------
    print("\n[5] After actor PPO-style PG-OPD update on same fixed rollout batch")

    after = opd_pg_eval_stats_teacher_forced_verl_close(
        student_policy=student_policy,
        tokenizer=tokenizer,
        rollout_items=rollout_items,
        clip_ratio_low=clip_ratio_low,
        clip_ratio_high=clip_ratio_high,
    )

    print("    loss:", after["loss"])
    print("    objective:", after["objective"])
    print("    mean_ratio:", after["mean_ratio"])
    print("    mean_rollout_is_weight:", after["mean_rollout_is_weight"])
    print("    mean_teacher_signal:", after["mean_teacher_signal"])
    print("    mean_clipfrac:", after["mean_clipfrac"])
    print("    per_sequence_objective:", after["per_sequence_objective"])

    print("\n[6] Discard this fixed rollout batch")
    print("    Next prompt creates fresh π_rollout and π_old from updated student.")


TEACHER-FORCED VERL-CLOSE PG-OPD EXAMPLE 1
Prompt: 'The capital of France is'
Completions:
    ' Paris.'
    ' London.'

[1] Policy roles
    π_rollout: frozen behavior policy for rollout tokens
    π_old:     frozen proximal/training-old policy
    πθ:        current trainable student_policy
    ν:         frozen teacher_policy
    In this HF demo, π_rollout and π_old are identical.
    In verl/vLLM-style systems, they may differ.

[2] Collect teacher-forced OPD rollout items once
    item 0:
        completion: ' Paris.'
        response_len: 2
        teacher_signal mean/min/max: -0.009244441986083984 -0.01665496826171875 -0.0018339157104492188
        rollout_is_weight mean/min/max: 1.0 1.0 1.0
    item 1:
        completion: ' London.'
        response_len: 2
        teacher_signal mean/min/max: -0.005862712860107422 -0.010259628295898438 -0.0014657974243164062
        rollout_is_weight mean/min/max: 1.0 1.0 1.0

[3] Before actor PPO-style PG-OPD update
    loss: 0.00755357742309